In [ ]:
import numpy as np 
import pandas as pd 

In [ ]:
df1 = pd.read_csv('/Users/ujjwalraj/Desktop/Book-Recomendation-System/Audible_Catlog.csv')
df2 = pd.read_csv('/Users/ujjwalraj/Desktop/Book-Recomendation-System/Audible_Catlog_Advanced_Features.csv')

In [ ]:
df1.info()

In [ ]:
df2.info()

In [ ]:
# Merging the Dataset on 'Book Name' and 'Author'
merge_df = pd.merge(df1, df2, on = ['Book Name', 'Author'], suffixes = ('_df1','_df2'))

In [ ]:
merge_df.head(3)

In [ ]:
merge_df.info()

In [ ]:
merge_df.shape


In [ ]:
# keep the 'Rating_df1', 'Number of Reviews_df1', and 'Price_df1' and drop the others
merge_df.drop(['Rating_df2', 'Number of Reviews_df2', 'Price_df2'], axis=1, inplace=True)

In [ ]:
# Rename kept columns for clarity
merge_df.rename(columns={
    'Rating_df1': 'Rating',
    'Number of Reviews_df1': 'Number of Reviews',
    'Price_df1': 'Price'
}, inplace=True)

In [ ]:
# Drop records with missing critical fields
merge_df.dropna(subset=['Rating', 'Number of Reviews', 'Price', 'Description'], inplace=True)

In [ ]:
merge_df

In [ ]:
# 1. Pull out the numbers that appear right before "hour" or "minute"
hours = merge_df['Listening Time'].str.extract(r'(\d+)\s*hour').fillna(0).astype(int)
minutes = merge_df['Listening Time'].str.extract(r'(\d+)\s*minute').fillna(0).astype(int)

# 2. Math it out into your brand new column
merge_df['Listening Time (Minutes)'] = (hours * 60) + minutes

In [ ]:
merge_df = merge_df.drop(columns = 'Listening Time')

In [ ]:
# Remove duplicates
merge_df.drop_duplicates(subset=['Book Name', 'Author'], inplace=True)

In [ ]:
merge_df.info()

In [ ]:
Simple_Rank_genre = merge_df['Ranks and Genre'].dropna().unique()[0:10]
Simple_Rank_genre

In [ ]:
import re 

def extract_exact_genre(rank_str):
    try:
        genre = re.findall(r'#\d+\s+in\s+([^(,]+)', rank_str)
        
        filtered = [g.strip()for g in genre if 'Audible Audiobooks & Originals' not in g]
        return filtered[0] if filtered else None
    except:
        return None 
    
merge_df['Genre'] = merge_df['Ranks and Genre'].apply(extract_exact_genre)
        
genre_counts = merge_df['Genre'].value_counts()
merge_df[['Book Name', 'Genre']].head(), genre_counts

In [ ]:
merge_df

In [ ]:
import re

def extract_best_rank(rank_str):
    ranks = re.findall(r'#(\d+)\s+in\s+([^(,]+)', rank_str)
    if not ranks:
        return None
    # Find the minimum rank value and corresponding genre
    best = min(ranks, key=lambda x: int(x[0]))
    return int(best[0])  # Return rank number

merge_df['Rank'] = merge_df['Ranks and Genre'].apply(extract_best_rank)

In [ ]:
merge_df.info()

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

known_genres = merge_df[merge_df['Genre'].notnull()]
unknown_genres = merge_df[merge_df['Genre'].isnull()]

known_genres = known_genres.dropna(subset=['Description'])
unknown_genres = unknown_genres.dropna(subset=['Description'])

if len(unknown_genres) == 0:
    print("🎉 No Missing Genres to Fill")
else:
    print(f"🔮 Found {len(unknown_genres)} missing genres. Predicting via K-NN...")

    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    tfidf_known = vectorizer.fit_transform(known_genres['Description'])
    tfidf_unknown = vectorizer.transform(unknown_genres['Description'])

    nn = NearestNeighbors(n_neighbors=1, metric='cosine')
    nn.fit(tfidf_known)

    distance, index = nn.kneighbors(tfidf_unknown, n_neighbors=1)
    predicted_Genre = known_genres.iloc[index.flatten()]['Genre'].values

    predictions_series = pd.Series(predicted_Genre, index=unknown_genres.index)
    merge_df['Genre'] = merge_df['Genre'].fillna(predictions_series)
    print("✅ Missing genres successfully filled!")

filled_genre_counts = merge_df['Genre'].value_counts().head(10)
print("\n--- SAMPLE ROWS ---")
print(merge_df[['Book Name', 'Genre']].head())
print("\n--- TOP 10 GENRES ---")
print(filled_genre_counts)

In [ ]:
merge_df.isnull().sum()

In [ ]:
merge_df['Rank'] = merge_df['Rank'].fillna(999)

merge_df['Rank'].isnull().sum(), merge_df['Rank'].value_counts().head()

In [ ]:
# Convert 'Number of Reviews' and 'Rank' to integer type
merge_df['Number of Reviews'] = merge_df['Number of Reviews'].astype(int)
merge_df['Rank'] = merge_df['Rank'].astype(int)

In [ ]:
# Convert 'Genre' and 'Ranks and Genre' to category dtype
merge_df['Genre'] = merge_df['Genre'].astype('category')
merge_df['Ranks and Genre'] = merge_df['Ranks and Genre'].astype('category')

In [ ]:
merge_df.info()

In [ ]:
merge_df.isnull().sum()

In [ ]:
merge_df = merge_df.drop(columns=['Ranks and Genre'])

In [ ]:
merge_df.head(5)

In [ ]:
merge_df.to_csv(r"/Users/ujjwalraj/Desktop/Book-Recomendation-System/Clean_Data.csv",index=False)

# Performing EDA on Clean_Data

In [ ]:
df = pd.read_csv('/Users/ujjwalraj/Desktop/Book-Recomendation-System/Clean_Data.csv')
df

In [ ]:
df.isnull().any().sum()

In [ ]:
df.info()

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

# Distribution of ratings across genres
rating_distribution = df.groupby('Genre')['Rating'].mean().reset_index()
rating_distribution = rating_distribution.sort_values(by='Rating', ascending=False)

top_genres = rating_distribution.head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='Rating', y='Genre', data=top_genres, palette='viridis')
plt.xticks(rotation=45, ha="right")
plt.title('Top 10 Average Rating Distribution Across Genres')
plt.xlabel('Average Rating')
plt.ylabel('Genre')

plt.tight_layout()  
plt.show()

In [ ]:
# Most common genres and authors.
genre_counts = df['Genre'].value_counts().head(10).reset_index()
genre_counts.columns = ['Genre', 'Count']

# Plot 
plt.figure(figsize=(12, 6))
sns.barplot(x='Count', y='Genre', data=genre_counts, palette='viridis')
plt.xticks(rotation=45, ha="right")
plt.title('Popular Genres in the Dataset')
plt.xlabel('Count')
plt.ylabel('Genre')

plt.tight_layout()
plt.show()

In [ ]:
# Relationship between book ratings and review counts.
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df,
                x='Rating',
                y='Number of Reviews',
                alpha=0.6,
                color='teal',
                edgecolor='w', s=60)

plt.title('Relationship Between Book Ratings and Review Counts', fontsize=14, pad=15)
plt.xlabel('Rating (out of 5)', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# What are the most popular genres in the dataset?
popular_genre = df['Genre'].value_counts().reset_index()
popular_genre.columns = ['Genre','Count']
# Showing Top 10 Genre
top_10 = popular_genre.head(10)
print(top_10)

In [ ]:
# Group by Author and calculate both the average rating and book count
author_stats = df.groupby('Author').agg(
    Average_Rating=('Rating', 'mean'),
    Book_Count=('Book Name', 'count')
).reset_index()

established_authors = author_stats[author_stats['Book_Count'] >= 3]

top_authors = established_authors.sort_values(by='Average_Rating', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='Average_Rating', y='Author', data=top_authors, palette='coolwarm')

plt.title('Top 10 Highest-Rated Authors (Minimum 3 Books Written)', fontsize=14, pad=15)
plt.xlabel('Average Rating', fontsize=12)
plt.ylabel('Author', fontsize=12)
plt.xlim(4.0, 5.0) 

plt.tight_layout()
plt.show()

In [ ]:
# What is the average rating distribution across books?
plt.figure(figsize=(10, 6))
# Plot a histogram with a smooth density line
sns.histplot(df['Rating'], bins=20, kde=True, color='royalblue', edgecolor='black')
mean_rating = df['Rating'].mean()
plt.axvline(mean_rating, color='red', linestyle='--', linewidth=2, label=f'Mean Rating: {mean_rating:.2f}')

plt.title('Distribution of Average Book Ratings', fontsize=14, pad=15)
plt.xlabel('Book Rating (out of 5)', fontsize=12)
plt.ylabel('Number of Books (Count)', fontsize=12)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# How do ratings vary between books with different review counts?
bins = [0, 50, 200, 500, 1000, 5000, 20000, df['Number of Reviews'].max()]
labels = ['0-50', '51-200', '201-500', '501-1000', '1001-5000', '5001-20000', '20001+']

df['Review Count Range'] = pd.cut(df['Number of Reviews'], bins=bins, labels=labels)
plt.figure(figsize=(12, 6))
sns.boxplot(x='Review Count Range', y='Rating', data=df, palette='Set2')

plt.title('Variation of Ratings with Number of Reviews')
plt.xlabel('Review Count Range')
plt.ylabel('Rating')

plt.tight_layout()
plt.show()

# Performing NLP and CLustering 

In [ ]:
import pandas as pd
df_clean = pd.read_csv('/Users/ujjwalraj/Desktop/Book-Recomendation-System/Clean_Data.csv')
df_clean.head()

In [ ]:
print(df_clean.duplicated().sum())
print(df_clean.info())

# Preprocess Text Column 

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split() # Tokenization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

In [ ]:
df_clean['combined_text'] = df_clean['Book Name'] + ' ' + df['Description']
df_clean['cleaned_text'] = df_clean['combined_text'].apply(preprocess_text)
df_clean['cleaned_text'].head()

In [ ]:
# Feature Extraction(TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)
X = tfidf.fit_transform(df_clean['cleaned_text'])

In [ ]:
tfidf

In [ ]:
X

In [ ]:
# Clustering Books with K-mean
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [ ]:
# Elbow Meothod to Choose the Value of K
inertia = []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X)
    inertia.append(km.inertia_)

In [ ]:
plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal k')
plt.show()

In [ ]:
# Fit model with chosen k
kmeans = KMeans(n_clusters=4, random_state=42)
df_clean['cluster'] = kmeans.fit_predict(X)

In [ ]:
df_clean['cluster'].value_counts()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
components = pca.fit_transform(X.toarray())

plt.figure(figsize=(10, 6))
plt.scatter(components[:, 0], components[:, 1], c=df_clean['cluster'], cmap='tab10')
plt.title('Book Clusters Based on Text Similarity')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.colorbar(label='Cluster ID')
plt.show()

In [ ]:
df_clean.head()

In [ ]:
df_clean.to_csv('/Users/ujjwalraj/Desktop/Book-Recomendation-System/Cluster_cleaned.csv',index = False)

# Training Models

In [ ]:
model_data = pd.read_csv('/Users/ujjwalraj/Desktop/Book-Recomendation-System/Cluster_cleaned.csv')
model_data.head(5)

# Content Based Recomendation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import joblib

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(model_data['cleaned_text'])

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
book_indices = pd.Series(model_data.index, index = model_data['Book Name']).drop_duplicates()

In [ ]:
joblib.dump(tfidf, '/Users/ujjwalraj/Desktop/Book-Recomendation-System/tfidf.pkl')
joblib.dump(cosine_sim, '/Users/ujjwalraj/Desktop/Book-Recomendation-System/cosine_sim_matrix.pkl')

In [ ]:
def Get_Content_Based_Recomendation(title, top_n = 5):
    idx = book_indices.get(title)
    
    if idx is None:
        return 'Book Not Found'
    sim_score = list(enumerate(cosine_sim[idx]))
    sim_score = sorted(sim_score, key=lambda x: x[1], reverse=True)[1:top_n+1]
    book_indices_top = [i[0] for i in sim_score]
    return model_data[['Book Name', 'Author', 'Genre']].iloc[book_indices_top]

In [ ]:
recomendation = Get_Content_Based_Recomendation('Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now')
recomendation

# Clustering Based recomendation

In [ ]:
cluster_map = model_data.set_index('Book Name')['cluster'].to_dict()

In [ ]:
def Get_Cluster_Based_Recomendation(book_title, top_n =5):
    if book_title not in cluster_map:
        return 'Book Not Found'
    
    cluster_id = cluster_map[book_title]
    cluster_books = model_data[model_data['cluster'] == cluster_id]
    recommendations = cluster_books[cluster_books['Book Name'] != book_title]
    return recommendations[['Book Name', 'Author', 'Genre']].head(top_n)

In [ ]:
book_title = "Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now"
Get_Cluster_Based_Recomendation(book_title)

# Hybrid Based Recomendation System

In [ ]:
def Get_Hybrid_Recomendation(title, top_n =5):
    idx = book_indices.get(title)
    if idx is None:
        return "Book not found."
    
    # Step 1: Content-based similarity
    sim_score = list(enumerate(cosine_sim[idx]))
    sim_score = sorted(sim_score, key=lambda x: x[1], reverse=True)[1:]  # Skip self

    # Step 2: Fetch books from same cluster
    input_cluster = model_data.loc[idx, 'cluster']

    # Step 3: Score boost based on cluster match and rating
    recommendations = []
    for i, sim in sim_score:
        cluster_score = 1.2 if model_data.loc[i, 'cluster'] == input_cluster else 1.0
        rating_score = model_data.loc[i, 'Rating']
        final_score = sim * cluster_score * rating_score
        recommendations.append((i, final_score))

    # Step 4: Sort and return top N
    top_recommendations = sorted(recommendations, key=lambda x: x[1], reverse=True)[:top_n]
    top_indices = [i[0] for i in top_recommendations]
    return model_data[['Book Name', 'Author', 'Genre', 'Rating']].iloc[top_indices]

In [ ]:
Get_Hybrid_Recomendation("Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now")

# StreamLit App Code

In [1]:
%%writefile streamlit_book_recomendation.py
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
import os
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# Set the page config with improved aesthetics
st.set_page_config(page_title="📚 Book Recommender", layout="wide")

@st.cache_data
def load_data():
    
    return pd.read_csv(r"/Users/ujjwalraj/Desktop/Book-Recomendation-System/Cluster_cleaned.csv")

@st.cache_resource
def load_models():
    
    tfidf = joblib.load(r"/Users/ujjwalraj/Desktop/Book-Recomendation-System/tfidf.pkl")
    cosine_sim = joblib.load(r"/Users/ujjwalraj/Desktop/Book-Recomendation-System/cosine_sim_matrix.pkl")
    return tfidf, cosine_sim

df = load_data()
tfidf, cosine_sim = load_models()

book_indices = pd.Series(df.index, index=df['Book Name']).drop_duplicates()
cluster_map = df.set_index('Book Name')['cluster'].to_dict()

# ---------------- Recommendation Functions ---------------- #

def get_content_based_recommendations(title, top_n=5):
    
    if title not in book_indices:
        return pd.DataFrame()
    
    idx = book_indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    book_indices_similar = [i[0] for i in sim_scores]
    return df.iloc[book_indices_similar][['Book Name', 'Author', 'Genre', 'Rating']]

def get_clustering_based_recommendations(preference_type, preference_input, top_n=5):
    
    if preference_type == "Genre":
        cluster_books = df[df['Genre'] == preference_input]
        
    else:
        cluster_books = df[df['Author'] == preference_input]

    return cluster_books.sort_values(by='Rating', ascending=False).head(top_n)

def get_hybrid_recommendations(preference_type, preference_input, top_n=5):
    
    filtered = df[df[preference_type] == preference_input]
    return filtered.sort_values(by='Rating', ascending=False).head(top_n)

def plot_wordcloud(text, title="Word Cloud"):
    
    stopwords = set(STOPWORDS)
    wordcloud = WordCloud(
        background_color='white',
        stopwords=stopwords,
        max_words=100,
        colormap='viridis',
        width=800,
        height=400
    ).generate(text)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    st.pyplot(fig)

# ---------------- Sidebar Navigation ----------------
page = st.sidebar.selectbox("Navigate", ["🏠 Home", "📊 EDA", "🔍 Recommendation System"])

# ---------------- HOME ----------------
if page == "🏠 Home":
    
    st.markdown("<h1 style='text-align: center; color: #4A4A4A;'>📚 Welcome to Your Personal Book Recommender</h1>", unsafe_allow_html=True)
    st.markdown("""
    <div style='text-align: center; font-size: 18px; padding: 10px 0; color: #5E5E5E;'>
    Discover books tailored to your taste and interests using smart algorithms.
    </div>
    """, unsafe_allow_html=True)
    st.markdown("---")
    st.markdown("### ✅ What You Can Do:")
    st.markdown("""
    - 🔍 **Explore books** by genre, author, or book title.
    - 📈 **Visualize trends** with interactive EDA.
    - 🤖 **Get smart recommendations** using content, clustering & hybrid models.
    """)
    st.markdown("---")

# ---------------- EDA ----------------
elif page == "📊 EDA":
    
    st.title("📊 Exploratory Data Analysis (EDA)")
    st.markdown("---")

    st.markdown("## 🟢 Easy Level Insights")

    # Most popular genres
    genre_counts = df['Genre'].value_counts().head(10).reset_index()
    genre_counts.columns = ['Genre', 'Count']
    st.plotly_chart(px.bar(genre_counts, x='Genre', y='Count', title='Top 10 Most Popular Genres'), use_container_width=True)

    # Top authors by rating
    top_authors = df.groupby('Author')['Rating'].mean().sort_values(ascending=False).head(10).reset_index()
    st.plotly_chart(px.bar(top_authors, x='Author', y='Rating', title='Top 10 Authors by Average Book Rating'), use_container_width=True)

    # Rating distribution
    st.plotly_chart(px.histogram(df, x='Rating', nbins=20, title='Average Rating Distribution'), use_container_width=True)

    # Reviews vs Ratings
    st.plotly_chart(px.scatter(df, x='Number of Reviews', y='Rating', 
                                title='Ratings vs Number of Reviews', trendline='ols'), use_container_width=True)

    st.markdown("---")
    st.markdown("## 🟡 Medium Level Insights")

    # Cluster examples
    cluster_books = df.groupby('cluster')['Book Name'].apply(lambda x: ', '.join(x.head(3))).reset_index()
    st.write("📚 **Top Books in Each Cluster:**")
    st.dataframe(cluster_books.rename(columns={'Book Name': 'Top Books in Cluster'}))

    genre_rating = df.groupby('Genre')['Rating'].mean().sort_values(ascending=False).head(10).reset_index()
    st.plotly_chart(px.bar(genre_rating, x='Genre', y='Rating', title='Genre Similarity and Average Ratings'), use_container_width=True)

    st.markdown("---")
    st.markdown("## 🔴 Scenario-Based Insights")

    sci_fi_recs = df[df['Genre'].str.contains("science fiction", case=False)].sort_values(by='Rating', ascending=False).head(5)
    st.write("👽 **Top 5 Science Fiction Books:**")
    st.dataframe(sci_fi_recs[['Book Name', 'Author', 'Rating']])

    hidden_gems = df[df['Number of Reviews'] < 50].sort_values(by='Rating', ascending=False).head(5)
    st.write("💎 **Hidden Gems (High Rating, Low Reviews):**")
    st.dataframe(hidden_gems[['Book Name', 'Author', 'Rating', 'Number of Reviews']])

    st.markdown("### 🧠 Word Cloud Generator")
    genre_choice = st.selectbox("Select Genre for Word Cloud", df['Genre'].dropna().unique())
    if st.button("Generate Word Cloud"):
        genre_text = " ".join(df[df['Genre'] == genre_choice]['cleaned_text'].dropna().astype(str))
        plot_wordcloud(genre_text, f"{genre_choice} Word Cloud")

# ---------------- Recommendation System ----------------
elif page == "🔍 Recommendation System":
    st.title("🔍 Personalized Book Recommendation System")
    st.markdown("---")

    st.subheader("🎯 Choose Your Preferences")
    preference_type = st.radio("Choose a Preference Filter", ["Genre", "Author", "Book Name"])
    
    if preference_type == "Genre":
        preference_input = st.selectbox("Select a Genre", sorted(df['Genre'].dropna().unique()))
    elif preference_type == "Author":
        preference_input = st.selectbox("Select an Author", sorted(df['Author'].dropna().unique()))
    else:
        preference_input = st.selectbox("Select a Book", sorted(df['Book Name'].dropna().unique()))

    top_n = st.slider("Number of Recommendations", min_value=3, max_value=10, value=5)

    st.subheader("🔢 Choose Recommendation Method")
    rec_type = st.radio("Select Recommendation Method", ["Clustering-Based", "Content-Based", "Hybrid"])

    if st.button("📚 Get Recommendations"):
        with st.spinner("Finding the perfect reads..."):
            if rec_type == "Clustering-Based":
                result = get_clustering_based_recommendations(preference_type, preference_input, top_n)
            elif rec_type == "Content-Based":
                result = get_content_based_recommendations(preference_input, top_n)
            else:
                result = get_hybrid_recommendations(preference_type, preference_input, top_n)

        if not result.empty:
            st.success("✅ Here are your recommended books:")

            for i in range(0, len(result), 2):
                cols = st.columns(2)
                for col_index in range(2):
                    if i + col_index < len(result):
                        book = result.iloc[i + col_index]
                        with cols[col_index]:
                            st.markdown(f"""
    <div style='
        background-color: #ffffff;
        padding: 20px;
        border-radius: 15px;
        box-shadow: 0px 0px 10px rgba(0, 0, 0, 0.1);
        margin-bottom: 20px;
        height: auto;
        color: #333333;
        font-family: sans-serif;
    '>
        <h4 style='color:#1a1a1a;'>{book['Book Name']}</h4>
        <p><strong>Author:</strong> {book['Author']}</p>
        <p><strong>Genre:</strong> {book.get('Genre', 'N/A')}</p>
        <p><strong>Rating:</strong> ⭐ {book['Rating']}</p>
    </div>
""", unsafe_allow_html=True)


            st.metric("📊 Average Rating", round(result['Rating'].mean(), 2))
            st.metric("📚 Total Recommendations", len(result))

            csv = result.to_csv(index=False).encode('utf-8')
            st.download_button("📥 Download Recommendations", csv, "recommendations.csv", "text/csv")
        else:
            st.warning("⚠️ No recommendations found based on your input.")

Overwriting streamlit_book_recomendation.py


In [ ]:
!streamlit run streamlit_book_recomendation.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.29.80:8501

